# Sparkov M3 — CatBoost Pseudo-Labelling (Corrected Model Selection)

**Selection rule:** The highest **validation PR-AUC (average precision)** wins at every stage. Notebook 05 identifies **M1 cost-sensitive CatBoost** as the highest-performing M0–M2 configuration (approximately 0.9892). This notebook uses that exact CatBoost hyperparameter family, **not LightGBM**.

**Important experimental distinction:** M1 is a *static* classifier trained on the original training split; M3 is a *weekly retrained* CatBoost classifier incorporating confidence-ranked pseudo-labels. Thus the M1-to-M3 comparison measures the combined effect of temporal retraining and pseudo-labelling, not pseudo-labelling in isolation. We additionally report a verified-only weekly CatBoost control to isolate the incremental contribution of pseudo-labels.

The 16-configuration pseudo-label grid and seven-day verification delay follow the archived M3 methodology. **M3 is selected for the next stage only if its held-out validation PR-AUC exceeds M1 CatBoost.** No test rows are used here.

In [1]:
from pathlib import Path
from itertools import product
import time
import json
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/sparkov/m3_catboost_selected_m1")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load the processed feature table and the pre-approved modelling feature list.
df = pd.read_pickle(DATA_DIR / "sparkov_features.pkl")
with open(DATA_DIR / "feature_lists.json", "r") as f:
    feature_lists = json.load(f)
feature_cols = feature_lists["sparkov_feature_cols"]

TIME_COL = "trans_date_trans_time"
TARGET_COL = "is_fraud"
SPLIT_COL = "sparkov_split"

# The test partition is excluded. Retain chronological train and validation only.
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
# Preserve the original row order used in Notebook 05.
m3_df = df.loc[
    df[SPLIT_COL].isin(["train", "validation"])
].copy()
del df

# Preserve the processed feature values exactly as in Notebook 05's M1 baseline.
# Do not recompute frequency features here: doing so would change the comparison.

train_mask = m3_df[SPLIT_COL].eq("train")
X_train_schema = m3_df.loc[train_mask, feature_cols]
categorical_cols = X_train_schema.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Data shape:", m3_df.shape)
print("Features:", len(feature_cols))
print("Categorical features:", categorical_cols)
print("Fraud rates by split:")
display(m3_df.groupby(SPLIT_COL)[TARGET_COL].agg(["size", "mean"]))


Data shape: (1296675, 87)
Features: 62
Categorical features: ['category', 'gender', 'state']
Fraud rates by split:


,size,mean
sparkov_split,,
train,1037340,0.005753
validation,259335,0.005931


## 1. Temporal setup and CatBoost implementation

The reproduced M1 CatBoost benchmark uses 500 iterations, learning rate 0.1, depth 6, Logloss, balanced class weights, seed 100, and four CPU threads. Original processed features and training-row order are preserved to match Notebook 05.

M3 adds rank-based pseudo-labelling under a seven-day verification delay, with class-specific reliability weighting. Weekly verified-only CatBoost is evaluated separately as a control because M1 itself is static. The development search comprises eight configurations over three training-period weekly cutoffs; full validation remains unchanged.


In [2]:
# Temporal configuration for M3 weekly retraining
WINDOW_DAYS = None                 # expanding history
RETRAIN_EVERY_DAYS = 7
LABEL_DELAY_DAYS = 7
# Reduced 8-configuration development grid.
# Lycia's selection rule and class-specific reliability weights are retained.
FRAUD_THRESHOLDS = [0.15, 0.50]
LEGIT_THRESHOLDS = [0.25, 0.50]
FRAUD_RELIABILITIES = [0.15, 0.50]
LEGIT_RELIABILITIES = [0.30]


def make_catboost(auto_class_weights="Balanced"):
    """Exact M1 CatBoost hyperparameters from Notebook 05."""
    return CatBoostClassifier(
        iterations=500, learning_rate=0.1, depth=6,
        loss_function="Logloss", cat_features=categorical_cols,
        auto_class_weights=auto_class_weights,
        thread_count=4, random_seed=100, verbose=False,
        allow_writing_files=False,
    )

def prepare_features(frame):
    """Match Notebook 05 CatBoost categorical preprocessing exactly."""
    X = frame[feature_cols].copy()
    for col in categorical_cols:
        X[col] = X[col].fillna("Missing").astype(str)
    return X

def verified_training_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select verified labels available at this cutoff; use expanding history."""
    cutoff = pd.Timestamp(cutoff)
    latest_eligible = cutoff - pd.Timedelta(days=label_delay_days)
    mask = frame[SPLIT_COL].isin(allowed_splits) & frame[TIME_COL].lt(latest_eligible)
    return frame.loc[mask].copy()


def pseudo_candidate_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select rows in the seven-day label-delay interval before the cutoff."""
    cutoff = pd.Timestamp(cutoff)
    start = cutoff - pd.Timedelta(days=label_delay_days)
    mask = (
        frame[SPLIT_COL].isin(allowed_splits)
        & frame[TIME_COL].ge(start)
        & frame[TIME_COL].lt(cutoff)
    )
    return frame.loc[mask].copy()


def make_cutoffs(start, end, frequency_days=RETRAIN_EVERY_DAYS):
    return pd.date_range(
        start=pd.Timestamp(start),
        end=pd.Timestamp(end),
        freq=f"{frequency_days}D",
        inclusive="left",
    )


def precompute_pseudo_scores(
    frame,
    start,
    end,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Cache one verified-only model's scores for each cutoff's delay window."""
    cache = {}
    cutoffs = make_cutoffs(start, end, frequency_days)

    for i, cutoff in enumerate(cutoffs, start=1):
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        candidates = pseudo_candidate_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        if verified.empty or verified[TARGET_COL].nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")
        if candidates.empty:
            cache[cutoff] = pd.DataFrame(
                columns=["row_index", "fraud_probability"]
            )
            continue

        X_verified = prepare_features(verified)
        X_candidates = prepare_features(candidates)
        model = make_catboost(auto_class_weights="Balanced")
        model.fit(X_verified, verified[TARGET_COL].astype(int))
        fraud_probability = model.predict_proba(X_candidates)[:, 1]

        cache[cutoff] = pd.DataFrame({
            "row_index": candidates.index.to_numpy(),
            "fraud_probability": fraud_probability,
        })
        print(
            f"Cached {i}/{len(cutoffs)} cutoffs: {cutoff} "
            f"({len(verified):,} verified; {len(candidates):,} unlabeled candidates)",
            flush=True,
        )
    return cache


def balanced_class_weight_map(y_verified):
    """Balanced class weights based only on verified labels at this cutoff."""
    classes = np.sort(pd.Series(y_verified).astype(int).unique())
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=pd.Series(y_verified).astype(int).to_numpy(),
    )
    return {int(label): float(weight) for label, weight in zip(classes, weights)}


def summarize_predictions(predictions):
    y_true = predictions["y_true"].astype(int)
    y_score = predictions["y_score"].astype(float)
    return {
        "transactions": len(predictions),
        "fraud_cases": int(y_true.sum()),
        "pr_auc": average_precision_score(y_true, y_score),
        "roc_auc": roc_auc_score(y_true, y_score),
    }


In [3]:
def run_adaptive_catboost(
    frame,
    start,
    end,
    fraud_thres=None,
    legit_thres=None,
    fraud_reliability=0.0,
    legit_reliability=0.0,
    pseudo_score_cache=None,
    allowed_splits=("train",),
    scoring_split="train",
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Weekly CatBoost retraining; None fraud_thres gives verified-only control."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if start >= end:
        raise ValueError("start must be earlier than end")
    use_pseudo = fraud_thres is not None
    if use_pseudo:
        if legit_thres is None:
            raise ValueError("legit_thres is required for M3")
        if not 0.0 < fraud_thres <= 1.5 or not 0.0 < legit_thres < 1.0:
            raise ValueError("fraud_thres must be in (0, 1.5], legit_thres in (0, 1)")
        if not 0 <= fraud_reliability <= 1 or not 0 <= legit_reliability <= 1:
            raise ValueError("Reliability weights must be in [0, 1]")

    prediction_blocks = []
    for cutoff in make_cutoffs(start, end, frequency_days):
        next_cutoff = min(cutoff + pd.Timedelta(days=frequency_days), end)
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        score_block = frame.loc[
            frame[SPLIT_COL].eq(scoring_split)
            & frame[TIME_COL].ge(cutoff)
            & frame[TIME_COL].lt(next_cutoff)
        ].copy()
        if score_block.empty:
            continue
        y_verified = verified[TARGET_COL].astype(int)
        if verified.empty or y_verified.nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")

        pseudo_rows = frame.iloc[0:0].copy()
        pseudo_rows["_m3_target"] = pd.Series(dtype=int)
        pseudo_rows["_m3_reliability"] = pd.Series(dtype=float)
        if use_pseudo:
            if pseudo_score_cache is None or cutoff not in pseudo_score_cache:
                raise KeyError(f"Missing pseudo-label scores for {cutoff}")
            scored_candidates = pseudo_score_cache[cutoff]
            if not scored_candidates.empty:
                # Same rank-based top/bottom selection as Lycia's IEEE M3.
                p = scored_candidates["fraud_probability"].to_numpy()
                sorted_scores = np.sort(p)
                verified_rate = float(y_verified.mean())
                n_fraud = min(max(int(round(fraud_thres * verified_rate * len(p))), 1), len(p))
                n_legit = min(max(int(round(legit_thres * len(p))), 1), len(p))
                fraud_cutoff = sorted_scores[-n_fraud]
                legit_cutoff = sorted_scores[n_legit - 1]
                is_fraud = p >= fraud_cutoff
                is_legit = p <= legit_cutoff
                # Lycia's implementation gives fraud precedence when masks overlap.
                keep = is_fraud | is_legit
                chosen = scored_candidates.loc[keep]
                if not chosen.empty:
                    pseudo_rows = frame.loc[chosen["row_index"].astype(int).to_numpy()].copy()
                    pseudo_rows["_m3_target"] = is_fraud[keep].astype(int)
                    pseudo_rows["_m3_reliability"] = np.where(
                        is_fraud[keep], fraud_reliability, legit_reliability
                    )

        verified_rows = verified.copy()
        verified_rows["_m3_target"] = y_verified.to_numpy()
        fit_rows = pd.concat([verified_rows, pseudo_rows], axis=0)
        y_fit = fit_rows["_m3_target"].astype(int)

        # Per-row sample weight = class weight from verified labels × reliability.
        class_weights = balanced_class_weight_map(y_verified)
        class_component = y_fit.map(class_weights).to_numpy(dtype=float)
        reliability_component = np.concatenate([
            np.ones(len(verified_rows), dtype=float),
            pseudo_rows["_m3_reliability"].to_numpy(dtype=float),
        ])
        fit_sample_weight = class_component * reliability_component

        X_fit = prepare_features(fit_rows)
        X_score = prepare_features(score_block)
        # Class balance is already represented in fit_sample_weight, avoiding double weighting.
        model = make_catboost(auto_class_weights=None)
        model.fit(X_fit, y_fit, sample_weight=fit_sample_weight)
        scores = model.predict_proba(X_score)[:, 1]

        prediction_blocks.append(pd.DataFrame({
            "row_index": score_block.index.to_numpy(),
            "timestamp": score_block[TIME_COL].to_numpy(),
            "y_true": score_block[TARGET_COL].astype(int).to_numpy(),
            "y_score": scores,
            "cutoff": cutoff,
            "fraud_thres": fraud_thres,
            "legit_thres": legit_thres,
            "fraud_reliability": fraud_reliability,
            "legit_reliability": legit_reliability,
            "verified_rows": len(verified_rows),
            "pseudo_rows": len(pseudo_rows),
            "class_weight_legitimate": class_weights[0],
            "class_weight_fraud": class_weights[1],
        }))

    if not prediction_blocks:
        raise ValueError("No scoring rows found")
    predictions = pd.concat(prediction_blocks, ignore_index=True)
    if predictions["row_index"].duplicated().any():
        raise AssertionError("Duplicate scored rows")
    return predictions


## 2. Reproduce M1 static CatBoost on validation (selection benchmark)

Train **once** on the original training split, exactly as Notebook 05 does. This is the correct baseline for stage selection, rather than the old M2 LightGBM score. Stop and investigate if the baseline does not reproduce the Notebook 05 result to within rounding tolerance.

In [4]:
train_rows = m3_df.loc[m3_df[SPLIT_COL].eq("train")]
validation_rows = m3_df.loc[m3_df[SPLIT_COL].eq("validation")]
X_train_m1 = prepare_features(train_rows)
X_val_m1 = prepare_features(validation_rows)
m1_model = make_catboost(auto_class_weights="Balanced")
m1_model.fit(X_train_m1, train_rows[TARGET_COL].astype(int))
m1_validation_predictions = pd.DataFrame({
    "row_index": validation_rows.index.to_numpy(),
    "timestamp": validation_rows[TIME_COL].to_numpy(),
    "y_true": validation_rows[TARGET_COL].astype(int).to_numpy(),
    "y_score": m1_model.predict_proba(X_val_m1)[:, 1],
})
m1_metrics = summarize_predictions(m1_validation_predictions)
print("M1 CatBoost validation PR-AUC:", f"{m1_metrics['pr_auc']:.6f}")
# Notebook 05 reports approximately 0.9892 (rounded to four decimals).
assert np.isclose(m1_metrics["pr_auc"], 0.9892, atol=0.0001, rtol=0), (
    "M1 CatBoost did not reproduce Notebook 05's rounded result. "
    "Check processed features, categorical preprocessing and split."
)
m1_validation_predictions.to_csv(RESULTS_DIR / "m1_catboost_validation_predictions.csv", index=False)
print("M1 CatBoost baseline reproduced.")

M1 CatBoost validation PR-AUC: 0.989228
M1 CatBoost baseline reproduced.


In [8]:

# Cost of the 7-day label delay for static M1 CatBoost

validation_start = validation_rows[TIME_COL].min()

delayed_rows = train_rows.loc[
    train_rows[TIME_COL]
    < validation_start - pd.Timedelta(days=LABEL_DELAY_DAYS)
].copy()

print(f"Original M1 training rows: {len(train_rows):,}")
print(f"Delayed M1 training rows: {len(delayed_rows):,}")

m1_delayed_model = make_catboost(auto_class_weights="Balanced")
m1_delayed_model.fit(
    prepare_features(delayed_rows),
    delayed_rows[TARGET_COL].astype(int)
)

m1_delayed_scores = m1_delayed_model.predict_proba(X_val_m1)[:, 1]

m1_delayed_pr_auc = average_precision_score(
    validation_rows[TARGET_COL].astype(int),
    m1_delayed_scores
)

print(f"M1 CatBoost, 0-day delay: {m1_metrics['pr_auc']:.6f}")
print(f"M1 CatBoost, 7-day delay: {m1_delayed_pr_auc:.6f}")
print(
    f"PR-AUC reduction from delay: "
    f"{m1_metrics['pr_auc'] - m1_delayed_pr_auc:+.6f}"
)


Original M1 training rows: 1,037,340
Delayed M1 training rows: 1,023,767
M1 CatBoost, 0-day delay: 0.989228
M1 CatBoost, 7-day delay: 0.987167
PR-AUC reduction from delay: +0.002061


## 3. Reduced development tuning (8 configurations × 3 weekly cutoffs)

To limit CatBoost training time, the development search evaluates eight prespecified combinations over the final three weekly cutoffs **within the training partition**. Fraud and legitimate pseudo-labelling thresholds and fraud reliability vary; legitimate reliability is fixed at 0.30.

The best configuration is selected using **development PR-AUC**, then evaluated over the **complete held-out validation period** in Section 4. The final stage is selected strictly by the highest validation PR-AUC across the eligible CatBoost candidates, including the reproduced M1 baseline. This reduced grid is not an exhaustive search.


In [6]:

# Development window: 3 weekly cutoffs before validation.
train_end = m3_df.loc[
    m3_df[SPLIT_COL].eq("train"), TIME_COL
].max()

validation_start = m3_df.loc[
    m3_df[SPLIT_COL].eq("validation"), TIME_COL
].min()

DEV_END = min(
    train_end.floor("D") + pd.Timedelta(days=1),
    validation_start
)
DEV_START = DEV_END - pd.Timedelta(days=21)

print("Last training timestamp:", train_end)
print("First validation timestamp:", validation_start)
print("Development start:", DEV_START)
print("Development end:", DEV_END)

assert train_end < validation_start, (
    "Training and validation timestamps overlap."
)
assert DEV_END <= validation_start
assert len(make_cutoffs(DEV_START, DEV_END)) == 3
assert len(list(product(
    FRAUD_THRESHOLDS,
    LEGIT_THRESHOLDS,
    FRAUD_RELIABILITIES,
    LEGIT_RELIABILITIES
))) == 8

# Distinct cache for the reduced development search.
DEV_RESULTS_DIR = RESULTS_DIR / "reduced_dev_8x3"
DEV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEV_PSEUDO_CACHE = precompute_pseudo_scores(
    frame=m3_df,
    start=DEV_START,
    end=DEV_END,
    allowed_splits=("train",),
)


def config_id(fraud_thres, legit_thres, fraud_reliability, legit_reliability):
    return (f"f{fraud_thres:.2f}_l{legit_thres:.2f}_"
            f"fw{fraud_reliability:.2f}_lw{legit_reliability:.2f}")


def tune_m3(frame, overwrite=False):
    results_path = DEV_RESULTS_DIR / "m3_development_grid.csv"
    rows = []
    configs = list(product(FRAUD_THRESHOLDS, LEGIT_THRESHOLDS,
                           FRAUD_RELIABILITIES, LEGIT_RELIABILITIES))
    reference_rows = None

    for i, (ft, lt, fr, lr) in enumerate(configs, start=1):
        name = config_id(ft, lt, fr, lr)
        prediction_path = DEV_RESULTS_DIR / f"development_{name}.csv"
        print(f"\n[{i}/{len(configs)}] {name}", flush=True)
        started = time.perf_counter()

        if prediction_path.exists() and not overwrite:
            pred = pd.read_csv(prediction_path, parse_dates=["timestamp", "cutoff"])
        else:
            pred = run_adaptive_catboost(
                frame=frame, start=DEV_START, end=DEV_END,
                fraud_thres=ft, legit_thres=lt,
                fraud_reliability=fr, legit_reliability=lr,
                pseudo_score_cache=DEV_PSEUDO_CACHE,
                allowed_splits=("train",), scoring_split="train",
            )
            pred.to_csv(prediction_path, index=False)

        metrics = summarize_predictions(pred)
        row = {"config_id": name, "fraud_thres": ft, "legit_thres": lt,
               "fraud_reliability": fr, "legit_reliability": lr,
               **metrics, "mean_pseudo_rows_per_retrain": pred["pseudo_rows"].mean()}
        rows.append(row)
        pd.DataFrame(rows).to_csv(results_path, index=False)

        scored_rows = set(pred["row_index"].astype(int))
        if reference_rows is None:
            reference_rows = scored_rows
        else:
            assert scored_rows == reference_rows, f"Scored rows differ for {name}"

        print(f"PR-AUC={metrics['pr_auc']:.4f} | ROC-AUC={metrics['roc_auc']:.4f} | "
              f"Mean pseudo rows/retrain={row['mean_pseudo_rows_per_retrain']:.0f} | "
              f"Elapsed={time.perf_counter() - started:.1f}s", flush=True)

    return pd.DataFrame(rows).sort_values(
        ["pr_auc", "roc_auc"], ascending=[False, False]
    ).reset_index(drop=True)


development_results = tune_m3(m3_df)
display(development_results)
best_config = development_results.iloc[0].to_dict()
print("Selected configuration:")
display(pd.DataFrame([best_config]))
pd.DataFrame([best_config]).to_csv(
    RESULTS_DIR / "m3_best_config.csv", index=False
)


Last training timestamp: 2020-03-06 07:15:17
First validation timestamp: 2020-03-06 07:16:43
Development start: 2020-02-14 07:16:43
Development end: 2020-03-06 07:16:43
Cached 1/3 cutoffs: 2020-02-14 07:16:43 (987,916 verified; 11,822 unlabeled candidates)
Cached 2/3 cutoffs: 2020-02-21 07:16:43 (999,738 verified; 11,959 unlabeled candidates)
Cached 3/3 cutoffs: 2020-02-28 07:16:43 (1,011,697 verified; 12,070 unlabeled candidates)

[1/8] f0.15_l0.25_fw0.15_lw0.30
PR-AUC=0.9893 | ROC-AUC=0.9999 | Mean pseudo rows/retrain=2999 | Elapsed=700.1s

[2/8] f0.15_l0.25_fw0.50_lw0.30
PR-AUC=0.9894 | ROC-AUC=0.9999 | Mean pseudo rows/retrain=2999 | Elapsed=689.5s

[3/8] f0.15_l0.50_fw0.15_lw0.30
PR-AUC=0.9865 | ROC-AUC=0.9998 | Mean pseudo rows/retrain=5988 | Elapsed=657.7s

[4/8] f0.15_l0.50_fw0.50_lw0.30
PR-AUC=0.9879 | ROC-AUC=0.9998 | Mean pseudo rows/retrain=5988 | Elapsed=675.8s

[5/8] f0.50_l0.25_fw0.15_lw0.30
PR-AUC=0.9888 | ROC-AUC=0.9999 | Mean pseudo rows/retrain=3024 | Elapsed=676.5s


,config_id,fraud_thres,legit_thres,fraud_reliability,legit_reliability,transactions,fraud_cases,pr_auc,roc_auc,mean_pseudo_rows_per_retrain
0,f0.15_l0.25_fw0.50_lw0.30,0.15,0.25,0.50,0.3,37602,268,0.989449,0.999865,2999.293601
1,f0.15_l0.25_fw0.15_lw0.30,0.15,0.25,0.15,0.3,37602,268,0.989308,0.999890,2999.293601
2,f0.50_l0.25_fw0.15_lw0.30,0.50,0.25,0.15,0.3,37602,268,0.988803,0.999861,3023.654566
3,f0.50_l0.50_fw0.50_lw0.30,0.50,0.50,0.50,0.3,37602,268,0.988333,0.999861,6012.269161
4,f0.50_l0.25_fw0.50_lw0.30,0.50,0.25,0.50,0.3,37602,268,0.987984,0.999834,3023.654566
5,f0.15_l0.50_fw0.50_lw0.30,0.15,0.50,0.50,0.3,37602,268,0.987939,0.999792,5987.908196
6,f0.50_l0.50_fw0.15_lw0.30,0.50,0.50,0.15,0.3,37602,268,0.987885,0.999862,6012.269161
7,f0.15_l0.50_fw0.15_lw0.30,0.15,0.50,0.15,0.3,37602,268,0.986527,0.999795,5987.908196


Selected configuration:


,config_id,fraud_thres,legit_thres,fraud_reliability,legit_reliability,transactions,fraud_cases,pr_auc,roc_auc,mean_pseudo_rows_per_retrain
0,f0.15_l0.25_fw0.50_lw0.30,0.15,0.25,0.5,0.3,37602,268,0.989449,0.999865,2999.293601


## 4. Held-out validation and mandatory stage selection

Compare (1) the original **static M1 CatBoost**, (2) a **verified-only weekly CatBoost** control, and (3) the tuned **M3 weekly CatBoost with pseudo-labels** on identical validation transactions. The **highest validation PR-AUC across all three evaluated candidates** determines the winner for Notebook 08. Results are written to a new CatBoost-specific directory, leaving archived LightGBM outputs untouched.

In [7]:
validation_rows = m3_df.loc[m3_df[SPLIT_COL].eq("validation")]
VAL_START = validation_rows[TIME_COL].min()
VAL_END = (
    validation_rows[TIME_COL].max().floor("D")
    + pd.Timedelta(days=1)
)
print("Validation rows:", len(validation_rows))
print("Validation start:", VAL_START)
print("Validation end:  ", VAL_END)

VAL_PSEUDO_CACHE = precompute_pseudo_scores(
    frame=m3_df,
    start=VAL_START,
    end=VAL_END,
    allowed_splits=("train", "validation"),
)

# Verified-only weekly CatBoost control (not the M1 selection baseline).
verified_only_predictions = run_adaptive_catboost(
    frame=m3_df, start=VAL_START, end=VAL_END,
    fraud_thres=None, allowed_splits=("train", "validation"),
    scoring_split="validation",
)

m3_validation_predictions = run_adaptive_catboost(
    frame=m3_df, start=VAL_START, end=VAL_END,
    fraud_thres=float(best_config["fraud_thres"]),
    legit_thres=float(best_config["legit_thres"]),
    fraud_reliability=float(best_config["fraud_reliability"]),
    legit_reliability=float(best_config["legit_reliability"]),
    pseudo_score_cache=VAL_PSEUDO_CACHE,
    allowed_splits=("train", "validation"), scoring_split="validation",
)

expected_rows = set(validation_rows.index)
for name, pred in [("M1 static", m1_validation_predictions),
                   ("Verified-only weekly", verified_only_predictions),
                   ("M3 pseudo-label", m3_validation_predictions)]:
    assert len(pred) == len(expected_rows), f"{name}: missing validation rows"
    assert set(pred["row_index"].astype(int)) == expected_rows, f"{name}: row mismatch"

control_metrics = summarize_predictions(verified_only_predictions)
m3_metrics = summarize_predictions(m3_validation_predictions)
validation_comparison = pd.DataFrame([
    {"stage": "M1", "model": "CatBoost", "strategy": "Static, balanced verified labels", **m1_metrics},
    {"stage": "Control", "model": "CatBoost", "strategy": "Weekly, verified-only", **control_metrics},
    {"stage": "M3", "model": "CatBoost", "strategy": "Weekly, pseudo-labelled", **m3_metrics},
])
display(validation_comparison)

# Apply the stated rule to ALL evaluated candidate configurations.
# The verified-only weekly control is also eligible if it achieves the highest AP.
ranked = validation_comparison.sort_values("pr_auc", ascending=False).reset_index(drop=True)
winner = ranked.iloc[0]
selected_stage = str(winner["stage"])
selected_pr_auc = float(winner["pr_auc"])
print(f"SELECTED FOR NEXT STAGE: {selected_stage} CatBoost | validation PR-AUC={selected_pr_auc:.6f}")
print(f"M3 minus M1 PR-AUC: {m3_metrics['pr_auc'] - m1_metrics['pr_auc']:+.6f}")
print(f"M3 minus verified-only weekly PR-AUC: {m3_metrics['pr_auc'] - control_metrics['pr_auc']:+.6f}")

validation_comparison.to_csv(RESULTS_DIR / "m3_validation_comparison.csv", index=False)
verified_only_predictions.to_csv(RESULTS_DIR / "verified_only_weekly_predictions.csv", index=False)
m3_validation_predictions.to_csv(RESULTS_DIR / "m3_validation_predictions.csv", index=False)
pd.DataFrame([{
    "selected_stage": selected_stage, "model": "CatBoost",
    "selected_validation_pr_auc": selected_pr_auc,
    "m1_validation_pr_auc": m1_metrics["pr_auc"],
    "m3_validation_pr_auc": m3_metrics["pr_auc"],
    "verified_only_weekly_pr_auc": control_metrics["pr_auc"],
    "m3_fraud_thres": best_config["fraud_thres"],
    "m3_legit_thres": best_config["legit_thres"],
    "m3_fraud_reliability": best_config["fraud_reliability"],
    "m3_legit_reliability": best_config["legit_reliability"],
}]).to_csv(RESULTS_DIR / "selected_stage_after_m3.csv", index=False)


Validation rows: 259335
Validation start: 2020-03-06 07:16:43
Validation end:   2020-06-22 00:00:00
Cached 1/16 cutoffs: 2020-03-06 07:16:43 (1,023,767 verified; 13,573 unlabeled candidates)
Cached 2/16 cutoffs: 2020-03-13 07:16:43 (1,037,340 verified; 16,114 unlabeled candidates)
Cached 3/16 cutoffs: 2020-03-20 07:16:43 (1,053,454 verified; 16,087 unlabeled candidates)
Cached 4/16 cutoffs: 2020-03-27 07:16:43 (1,069,541 verified; 15,950 unlabeled candidates)
Cached 5/16 cutoffs: 2020-04-03 07:16:43 (1,085,491 verified; 15,855 unlabeled candidates)
Cached 6/16 cutoffs: 2020-04-10 07:16:43 (1,101,346 verified; 16,057 unlabeled candidates)
Cached 7/16 cutoffs: 2020-04-17 07:16:43 (1,117,403 verified; 15,984 unlabeled candidates)
Cached 8/16 cutoffs: 2020-04-24 07:16:43 (1,133,387 verified; 15,766 unlabeled candidates)
Cached 9/16 cutoffs: 2020-05-01 07:16:43 (1,149,153 verified; 15,872 unlabeled candidates)
Cached 10/16 cutoffs: 2020-05-08 07:16:43 (1,165,025 verified; 15,677 unlabeled c

,stage,model,strategy,transactions,fraud_cases,pr_auc,roc_auc
0,M1,CatBoost,"Static, balanced verified labels",259335,1538,0.989228,0.999875
1,Control,CatBoost,"Weekly, verified-only",259335,1538,0.987465,0.999797
2,M3,CatBoost,"Weekly, pseudo-labelled",259335,1538,0.988218,0.999845


SELECTED FOR NEXT STAGE: M1 CatBoost | validation PR-AUC=0.989228
M3 minus M1 PR-AUC: -0.001010
M3 minus verified-only weekly PR-AUC: +0.000753



## 5. Interpretation and Next-Stage Handoff

### 5.1 Validation Results

The selected M3 configuration was evaluated against static M1 CatBoost and a matched verified-only weekly CatBoost control. Both weekly models retrain every seven days under a seven-day label verification delay, with M3 additionally incorporating pseudo-labelled recent transactions.

| Model | Validation PR-AUC |
|---|---:|
| M1: Static CatBoost | **0.989228** |
| M2-style: Verified-only weekly CatBoost | 0.987465 |
| M3: Pseudo-labelled weekly CatBoost | 0.988218 |

M3 improved validation PR-AUC by **0.000753** over the matched verified-only weekly control, suggesting that pseudo-labelling can partially compensate for unavailable recent labels. However, M3 remained **0.001010 below static M1 CatBoost**.

### 5.2 Selected Pseudo-Labelling Configuration

The development search evaluated eight configurations across three chronological cutoffs. The selected parameters were:

| Parameter | Selected Value |
|---|---:|
| Fraud selection threshold | 0.15 |
| Legitimate selection threshold | 0.25 |
| Fraud pseudo-label reliability weight | 0.50 |
| Legitimate pseudo-label reliability weight | 0.30 |

The best development PR-AUC was **0.989449**. These parameters were selected using training-period development data before full validation.

### 5.3 Interpretation

The improvement over the verified-only weekly control indicates that incorporating recent unverified transactions through pseudo-labelling can provide a marginal benefit under the simulated seven-day verification delay.

Nevertheless, the improvement is small and has not been tested for robustness across multiple random seeds. Furthermore, Sparkov's relatively stable fraud patterns and already high baseline performance may limit the additional value of adaptive retraining and pseudo-labelling.

Overall, M3 demonstrates a modest improvement over its matched delayed-label control but does not outperform the strongest static model.

### 5.4 Final Model Selection and M4 Handoff

**M1 static CatBoost is retained as the selected model**, achieving the highest validation PR-AUC of **0.989228**.

Notebook 08 (M4) will therefore evaluate whether incorporating unsupervised anomaly information can improve upon this M1 CatBoost baseline.

The held-out test set remains reserved for final evaluation after model selection.
